In [2]:
import cobra
from cobra import Model, Reaction, Metabolite
import sys
sys.path.append('/home/nathan/public_thermo_flux/')
from thermo_flux.core.metabolite import ThermoMetabolite
import thermo_flux
from thermo_flux.io import load_excel as ex
from thermo_flux.core.model import ThermoModel, ThermoReaction
from equilibrator_api import  Q_
import pandas as pd
from thermo_flux.io import load_gams as gs
from thermo_flux.io import helper_load as hl
from thermo_flux.io import load_sbml as ls
from thermo_flux.utils.vis import compare_met
import numpy as np
import gurobipy as  gp
from gurobipy import GRB

import warnings


import os
import seaborn as sns
import matplotlib.pyplot as plt


## Notebook to test the fully automated part of the thermoflux pipeline on BIGG models 
In this notebook we load the stoichiometric models from BiGG and run Thermo-Flux to convert them to thermodynamic models. We use the main Thermo-Flux functions such as reaction_balance, update_thermo_info and add_TFBA_variables.

This notebook generates information about the number of reaction that could not be handled or balanced for each model, and also about FBA and TFBA optimization results.

In the same cell (to facilitate data generation) we will iterate on each models of the BIGG database and : 
- Load the model and identify models that cannot be loaded or have a zero cobra growth prediction
- balance the reactions with the thermo-flux tool and identify models that have reactions that can't be handled
- build the thermodynamic model and identify models that have reactions that are still not balanced (they need manual curation)
- compute the growth prediction within the thermodynamic constraints

In [3]:
# Create DFs to store model statistics
models_stats=pd.DataFrame(columns=['Status','stoichiometric optimization','Failed to balance','Biomass reactions','atom bag unbalanced rxns','Balance warnings','T-S optimization'])
not_handled_stats=pd.DataFrame(columns=['Model','N compartments','error'])
unbalanced_stats=pd.DataFrame(columns=['Model','error','rxn'])

In [3]:
#or load saved ones (from interrupted runs)
models_stats=pd.read_csv('models_stats_save.csv',index_col=0)
not_handled_stats=pd.read_csv('not_handled_stats_save.csv',index_col=0)
unbalanced_stats=pd.read_csv('unbalanced_stats_save.csv',index_col=0)

In [ ]:
# Get the list of models in the specified directory
files = [f for f in os.listdir('downloaded_models/') if os.path.isfile(os.path.join('downloaded_models', f))]

unbalanced_rxns = {}# will store unbalanced reactions for analysis

# Set the options for the analysis type
ignore_snd_unbalanced = True
remove_rxn_directions = False
cluster=False

# Loop through each model 
for model_name in files :    
    try: #loading is available from different formats
        tmodel=ls.load_model('downloaded_models/'+model_name)
        print(f'model   {model_name}   loaded')
        models_stats.loc[model_name,'Status']='Loaded'
    except:
        print(f'model   {model_name}   failed to load')
        models_stats.loc[model_name,'Status']='Not loaded'
        continue

    tmodel.objective = tmodel.reactions.biomass_EX#set objective
    #remove directions of reactions if needed
    if remove_rxn_directions:
        for rxn in tmodel.reactions:
            if 'EX_' not in rxn.id and 'biomas' not in rxn.id.lower():
                rxn.bounds=(-1000,1000)
    #first stats on the stoichiometric optimization 
    models_stats.loc[model_name,'stoichiometric optimization']=tmodel.optimize().to_frame().loc['biomass_EX']['fluxes']# not a yield 
    models_stats.loc[model_name,'Failed to balance']=0
    models_stats.loc[model_name,'atom bag unbalanced rxns']=0
    models_stats.loc[model_name,'Biomass reactions']=0

    #balance the reactions
    is_biomass=False#we will ignore the fact that biomass reactions may be still unbalanced
    for rxn in tmodel.reactions:
        if 'biomas' in rxn.id.lower():
            is_biomass=True
        else:
            is_biomass=False
        try:
            thermo_flux.tools.drg_tools.reaction_balance(rxn, balance_charge=True, balance_mg=False,round_dp=1)
        except Exception as e:
            print(f'failed to balance {rxn.id}')
            if not is_biomass:
                models_stats.loc[model_name,'Failed to balance']=models_stats.loc[model_name,'Failed to balance']+1
                not_handled_stats.loc[rxn.id,'Model']=model_name
                not_handled_stats.loc[rxn.id,'N compartments']=len(rxn.compartments)
                not_handled_stats.loc[rxn.id,'error']=e
                unbalanced_rxns[rxn.id]=rxn.reaction
                print(e)
            continue
        if not rxn.boundary:
            try:
                #we can also check the atom balance of each reactions and identify the ones that are not balanced
                dict=rxn.check_atom_balance()
                if dict != {}:
                    # print(f'atom bag not empty{rxn.id}')
                    # print(dict)
                    models_stats.loc[model_name,'atom bag unbalanced rxns']=int(models_stats.loc[model_name,'atom bag unbalanced rxns']+1)
            except Exception as e:
                print('atom bag check failed',e)
                continue
            
    #Building the thermodynamic part of the model
    tmodel._rmse_inf=Q_(1e3,'kJ/mol')
    try: #try to apply pipeline to the model, if it fails we skip it
        warning_count = 0
        #we catch the warnings occuring during the update_thermo_info
        ##some warnings will be about metabolite formulas
        #the interesting ones here are on reactions that are not identified as balanced (used further below)
        with warnings.catch_warnings(record=True) as w:
            tmodel.update_thermo_info(fit_unknown_dfG0=True,search=True)

            # Increment the warning count
            warning_count += len(w)
        # Initialize the warning count
        models_stats.loc[model_name,'Balance warnings']=warning_count
        models_stats.loc[model_name,'Status']='TF applied'

        #attributes of thermo model
        #ignore snd law for water transport and proton/charge transport
        for rxn in tmodel.reactions:
            #     #water transport is not included in snd law
            if 'H2O' in rxn.id and 'GLT' not in rxn.id:
                    print(rxn.id,rxn.reaction)
                    rxn.ignore_snd=True
            if all([met in tmodel.charge_dict.values() or met in tmodel.proton_dict.values() for met in rxn.metabolites]):
            #since we don't have pH or I values, proton and charge transport reaction energies can't be !=0 so we ignore 2nd law
                rxn.ignore_snd=True
                print('ignore_snd',rxn.id,rxn.ignore_snd)
            ##same for transport reactions - even if these could be directed only by metabolite concentrations. Here, not having information on biochemical parameters 
            #can prevent computing the correct metabolite concentrations
            if rxn.drG0prime.m ==0 and rxn.drGtransport.m==0:
                rxn.ignore_snd=True
        
        #we ignore the second law for reactions with very high uncertainty : numeric errors on the bound of qm prevents the drG error to match the drG0prime 
                #thus the reaction can't go in the right direction
            if abs(rxn.drG0prime.m)>tmodel._rmse_inf.m:
                rxn.ignore_snd=True
                print('ignore_snd for',rxn.id,rxn.ignore_snd)
        #ignore_concentration for protons, charges and biomass metabolites
        for met in tmodel.metabolites:
            if met in tmodel.proton_dict.values() or met in tmodel.charge_dict.values() or met.biomass==True:
                print(met.id, met.ignore_conc)
                met.ignore_conc = True

        ## part to ignore snd law of unbalanced reactions found in update_thermo_info : 
        #use the list of warnings to get the id of the reactions (as a string)
        if ignore_snd_unbalanced:
            rxn_to_ignore=[str(warning.message).split(' ')[0] for warning in w if 'Metabolite' not in str(warning.message)]
            if len(rxn_to_ignore)>0:
                unbalanced_stats.loc[model_name,'rxn']=rxn_to_ignore
                unbalanced_stats.loc[model_name,'error']=str(w)
                unbalanced_stats.loc[model_name,'Model']=model_name
                for rxn in [tmodel.reactions.get_by_id(rxn_id) for rxn_id in rxn_to_ignore]:
                    rxn.ignore_snd=True
                    print('ignore_snd for',rxn.id,rxn.ignore_snd)
            else:
                unbalanced_stats.loc[model_name,'rxn']='No unbalanced rxns'
                unbalanced_stats.loc[model_name,'Model']=model_name
            

        #gurobi solver TFBA part  
        # tmodel.reactions.biomass_EX.lower_bound=0.01 # if we want to force the flux to be non zero and analyse models with 0 biomass flux
        tmodel.m=None 
        tmodel.m.params.FeasibilityTol=1e-4
        tmodel.add_TFBA_variables(gdiss_constraint = False, sigmac_limit = (5000/tmodel.T.m), error_type = 'covariance',qnorm='sep_norm',alpha=0.90)
        # tmodel.m.update()
        # tmodel.m.params.MIPGap=1e-4
        if cluster: #if we want to run the optimization on the cluster
            tmodel.m.write('optim/'+model_name+'.mps')
        else :# optimize locally
            tmodel.m.params.NonConvex=-1
            tmodel.m.params.TimeLimit=60*5
            tmodel.m.params.OutputFlag=0 #no output shown
            tmodel.m.optimize()
            try : 
                models_stats.loc[model_name,'T-S optimization']=np.round(tmodel.m.objVal,3)
            except AttributeError:
                models_stats.loc[model_name,'T-S optimization']=0

    #if there is an error in the building the thermodynamic part of the model
    except Exception as e:
        models_stats.loc[model_name,'Status']='TF failed'
        print(e)
        continue
    # # #save the summary dfs';
    not_handled_stats.to_csv(f'not_handled_stats_save_fixed{remove_rxn_directions}.csv')
    unbalanced_stats.to_csv(f'unbalanced_stats_save_fixed{remove_rxn_directions}.csv')
    models_stats.to_csv(f'models_stats_save_fixed{remove_rxn_directions}.csv')

    
    

model   script_dl_models.sh   failed to load


Analysing specific models

In [14]:
tmodel=ls.load_model('downloaded_models/'+'iND750.sbml')

Initializing component contribution object...
cxcalc is not installed, operating in read-only mode. A local cache may be loaded, but no compounds can be created. Install cxcalc and obtain a ChemAxon license to enable compound creation.
Loading compounds from iND750_compound.sqlite
added reaction:  biomass_ce: biomass_c <=> biomass_e
added reaction:  biomass_EX: biomass_e <=> 
added reaction:  charge_ce: charge_e <=> charge_c
added reaction:  EX_charge: charge_e <=> 
added reaction:  charge_cm: charge_m <=> charge_c
added reaction:  charge_cx: charge_x <=> charge_c
added reaction:  charge_cn: charge_n <=> charge_c
added reaction:  charge_cg: charge_g <=> charge_c
added reaction:  charge_cv: charge_v <=> charge_c
added reaction:  charge_cr: charge_r <=> charge_c


## process the results and generate figures in notebook "Thermoflux_BIGGmodels_analysis_vf_260224"